# RSSM Evaluation

Thin notebook shell — all logic lives in `pim/world_models/rssm/run_eval.py`.

**Sections**
1. Setup & Quick Validation
2. Criterion 1 — Predictive Quality
3. Criterion 2 — Recovery
4. Criterion 3 — Rollout Consistency
5. Criterion 4 — Counterfactual Controllability

In [ ]:
import sys
sys.path.insert(0, ".")

import matplotlib.pyplot as plt
from IPython.display import display, HTML

import helpers.nb_viz as nb_viz
import pim.world_models.rssm.run_eval as eval
from pim.world_models.rssm.run_eval import _run_autoregressive_rssm
from pim.simulator.dataset import load_sample

import importlib
importlib.reload(nb_viz)
importlib.reload(eval)

## Config — edit these

In [ ]:
cfg = eval.EvalConfig(
    checkpoint_path = "../runs/9_dset4_rssm_0kl/best_model.pt",
    test_h5_path    = "../datasets/4_fixed_refl_inview/test.h5",
    edits_h5_path   = "../datasets/4_fixed_refl_inview/edits.h5",
    output_dir      = "../outputs/eval_rssm",
    device          = "cuda",
    batch_size      = 512,
    num_workers     = 6,
    # Criterion 1
    n_context       = 10,
    # Criterion 2
    n_obj           = 2,
    use_hungarian   = False,
    use_lstsq       = True,
    probe_n_epochs  = 30,
    probe_lr        = 5e-3,
    # Criterion 3
    rollout_n_context = 20,
    rollout_n_rollout = 20,
    coherence_n_eval  = 500,
    # Criterion 4
    ctrl_n_rollout    = 15,
)

---
## 1 — Setup & Quick Validation

In [ ]:
s = eval.setup(cfg)

In [ ]:
for fig in eval.plot_setup(cfg, s).values():
    display(fig)
    plt.close(fig)

---
## 2 — Criterion 1: Predictive Quality

In [ ]:
c1 = eval.run_criterion1(cfg, s)

In [ ]:
for fig in eval.plot_criterion1(cfg, s, c1).values():
    display(fig)
    plt.close(fig)

In [ ]:
# Waterfall pairs — actual vs imagination rollout (dark simulator aesthetic)
for i in range(3):
    scene, obs_depth, obs_id, obs_intensity = load_sample(cfg.test_h5_path, i)
    nb_viz.plot_waterfall_pair(
        obs_depth, obs_id, obs_intensity, scene,
        c1.obs_rollout[i], cfg.n_context,
        title=f"Sample {i}  —  actual vs imagined (warm-up={cfg.n_context})",
        dark=True,
    )
    plt.show()

---
## 3 — Criterion 2: Recovery

In [ ]:
c2 = eval.run_criterion2(cfg, s, c1)

In [ ]:
for fig in eval.plot_criterion2(cfg, s, c1, c2).values():
    display(fig)
    plt.close(fig)

---
## 4 — Criterion 3: Rollout Consistency

In [ ]:
c3 = eval.run_criterion3(cfg, s, c2)

In [ ]:
for fig in eval.plot_criterion3(cfg, s, c1, c3).values():
    display(fig)
    plt.close(fig)

---
## 5 — Criterion 4: Counterfactual Controllability

In [ ]:
c4 = eval.run_criterion4(cfg, s, c2)

In [ ]:
for fig in eval.plot_criterion4(cfg, s, c2, c4).values():
    display(fig)
    plt.close(fig)

---
## Appendix — 3-panel animation

In [ ]:
ANIM_SAMPLE_IDX = 0
ANIM_INTERVAL   = 80

scene, obs_depth, obs_id, obs_intensity = load_sample(cfg.test_h5_path, ANIM_SAMPLE_IDX)
pred, _ = _run_autoregressive_rssm(s.model, obs_intensity, cfg.n_context, cfg.device)

anim = nb_viz.animate_3panel(
    scene, obs_depth, obs_id, obs_intensity,
    pred, cfg.n_context,
    interval=ANIM_INTERVAL,
    title=f"Sample {ANIM_SAMPLE_IDX}  |  warm-up={cfg.n_context}  (RSSM imagination)",
    dark=True,
)
plt.close()
HTML(anim.to_jshtml())